# Resolução — S03-06

Notebook em **Julia**, feito para atender aos requisitos pedido pelo professor 
Prof. Kenny Vinente dos Santos, Dr. | kennyvinente@ufam.edu.br
# Nome: Renyer Montefusco levy
# Matrícula: 2240917

Resolução em Julia dos algoritmos pedidos no notebook do professor.


### Pacotes importados

In [1]:
using LinearAlgebra
using Printf

## Chapter 12 — Trust Region Methods

Nos métodos de região de confiança, aproximamos a função objetivo localmente por um modelo quadrático:

\[
q_k(d)=f(x_k)+\nabla f(x_k)^Td+\frac{1}{2}d^TB_kd
\]

e restringimos o passo:

\[
||d|| \leq \Delta_k
\]

onde \(\Delta_k\) é o raio da região de confiança.


# Algorithm 12.1 — Cauchy point

O ponto de Cauchy minimiza o modelo quadrático ao longo da direção do gradiente negativo.


In [2]:
function cauchy_point(g, B, Delta)

    gnorm = norm(g)

    if gnorm == 0
        return zeros(length(g))
    end

    gBg = dot(g, B*g)

    if gBg <= 0
        tau = 1.0
    else
        tau = min((gnorm^3)/(Delta*gBg), 1.0)
    end

    d = -(tau*Delta/gnorm) * g

    return d
end

cauchy_point (generic function with 1 method)

## Example 12.2

\[
g=
\begin{bmatrix}
1 \\
2
\end{bmatrix}
\]

\[
B=
\begin{bmatrix}
4 & 1 \\
1 & 3
\end{bmatrix}
\]

\[
\Delta=1
\]


In [3]:
g = [1.0, 2.0]

B = [
    4.0  1.0;
    1.0  3.0
]

Delta = 1.0

d_cauchy = cauchy_point(g, B, Delta)

println("Ponto de Cauchy:")
display(d_cauchy)

println("Norma do passo:")
println(norm(d_cauchy))

Ponto de Cauchy:


2-element Vector{Float64}:
 -0.25000000000000006
 -0.5000000000000001

Norma do passo:
0.5590169943749476


# Algorithm 12.2 — Dogleg method

O método Dogleg combina:
- direção de máximo declive;
- direção de Newton;
- restrição da trust region.


In [4]:
function dogleg_method(g, B, Delta)

    # Direção de Newton
    pB = -(B \ g)

    # Direção de steepest descent
    pU = -(dot(g,g)/dot(g,B*g)) * g

    if norm(pB) <= Delta
        return pB
    end

    if norm(pU) >= Delta
        return Delta * pU / norm(pU)
    end

    # Interpolação entre pU e pB
    a = dot(pB - pU, pB - pU)
    b = 2*dot(pU, pB - pU)
    c = dot(pU,pU) - Delta^2

    beta = (-b + sqrt(b^2 - 4*a*c)) / (2*a)

    return pU + beta*(pB - pU)
end

dogleg_method (generic function with 1 method)

## Teste do Dogleg

Usando os mesmos dados do Example 12.2.


In [5]:
d_dogleg = dogleg_method(g, B, Delta)

println("Passo Dogleg:")
display(d_dogleg)

println("Norma do passo:")
println(norm(d_dogleg))

Passo Dogleg:


2-element Vector{Float64}:
 -0.09090909090909091
 -0.6363636363636364

Norma do passo:
0.642824346533225


# Algorithm 12.3 — Trust region method

Agora implementamos o algoritmo completo de trust region.


In [6]:
function trust_region_method(f, gradf, Bfunc, x0;
    Delta0=1.0,
    eta=0.1,
    eps=1e-8,
    maxiter=100)

    x = float.(x0)
    Delta = Delta0

    historico = []

    for k in 0:maxiter

        g = gradf(x)
        B = Bfunc(x)

        push!(historico, (
            iter=k,
            x=copy(x),
            Delta=Delta,
            norma_gradiente=norm(g),
            valor_funcao=f(x)
        ))

        if norm(g) <= eps
            break
        end

        d = dogleg_method(g, B, Delta)

        ared = f(x) - f(x + d)

        pred = -(dot(g,d) + 0.5*dot(d,B*d))

        if abs(pred) < eps
            break
        end

        rho = ared/pred

        if rho < 0.25
            Delta *= 0.25
        elseif rho > 0.75 && abs(norm(d) - Delta) < 1e-10
            Delta = min(2*Delta, 100.0)
        end

        if rho > eta
            x = x + d
        end
    end

    return x, historico
end

function mostrar_historico_trust(historico)

    println("Iteração | x | Delta | ||grad|| | f(x)")
    println("--------------------------------------------------------------------------")

    for linha in historico

        @printf("%8d | [% .8f, % .8f] | %.6f | %.16e | %.16f\n",
            linha.iter,
            linha.x[1],
            linha.x[2],
            linha.Delta,
            linha.norma_gradiente,
            linha.valor_funcao
        )
    end
end

mostrar_historico_trust (generic function with 1 method)

## Aplicação na função de Rosenbrock

\[
f(x_1,x_2)=100(x_2-x_1^2)^2+(1-x_1)^2
\]


In [7]:
function rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return 100*(x2 - x1^2)^2 + (1 - x1)^2
end

function grad_rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return [
        -400*x1*(x2 - x1^2) - 2*(1 - x1),
        200*(x2 - x1^2)
    ]
end

function hess_rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return [
        1200*x1^2 - 400*x2 + 2   -400*x1;
        -400*x1                  200
    ]
end

hess_rosenbrock (generic function with 1 method)

## Execução do método trust region

Usando:

\[
x_0=
\begin{bmatrix}
-1.2 \\
1
\end{bmatrix}
\]


In [8]:
x0 = [-1.2, 1.0]

solucao, historico =
    trust_region_method(
        rosenbrock,
        grad_rosenbrock,
        hess_rosenbrock,
        x0;
        Delta0=1.0,
        maxiter=200
    )

mostrar_historico_trust(historico)

println()
println("Solução aproximada:")
display(solucao)

println("Valor da função:")
println(rosenbrock(solucao))

println("Norma do gradiente:")
println(norm(grad_rosenbrock(solucao)))

Iteração | x | Delta | ||grad|| | f(x)
--------------------------------------------------------------------------
       0 | [-1.20000000,  1.00000000] | 1.000000 | 2.3286768775422664e+02 | 24.1999999999999957
       1 | [-1.17528090,  1.38067416] | 1.000000 | 4.6394262140667566e+00 | 4.7318843252666083
       2 | [-1.17528090,  1.38067416] | 0.250000 | 4.6394262140667566e+00 | 4.7318843252666083
       3 | [-1.07407754,  1.15207433] | 0.500000 | 4.8321007248661667e+00 | 4.3020435568769040
       4 | [-0.85937237,  0.70051958] | 1.000000 | 1.8422478054768806e+01 | 3.6016754580212815
       5 | [-0.64317279,  0.36692898] | 1.000000 | 1.7939935032524485e+01 | 2.9185007047905174
       6 | [-0.48438839,  0.20941962] | 1.000000 | 9.3332439812577377e+00 | 2.2669758287846853
       7 | [-0.23873029, -0.00335575] | 1.000000 | 1.4614232831529776e+01 | 1.8986396233127463
       8 | [-0.14395063,  0.01173860] | 1.000000 | 3.3311859009568203e+00 | 1.3166928136779474
       9 | [-0.14395063,  0.01

2-element Vector{Float64}:
 0.9999983082930026
 0.99999659792968

Valor da função:
2.89668909123374e-12
Norma do gradiente:
5.529469774089269e-6
